NAME SAMIM ALI ROLL NO 24MDS031

In [1]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [2]:
X_train = ["Win a big prize now",
            "Limited time offer on prize",
            "Exclusive offer just for you",
            "Meeting at 5 PM today",
            "Lunch with friends at my place",
            "Join us for a meeting tomorrow"]

In [3]:
y_train = np.array(["Spam",
                    "Spam",
                    "Spam",
                    "Not Spam",
                    "Not Spam",
                    "Not Spam"])

In [4]:
X_test = ["Win a prize today",
          "Join us for lunch",
          "Exclusive limited time offer",
          "Meeting at my place tomorrow"]

In [5]:
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

In [6]:
stopwords = {"a", "the", "is"}
unique_words = np.array(sorted(i for i in {word for msg in X_train for word in msg.lower().split()} if i not in stopwords and not i.isnumeric()))

word_matrix = np.array([[msg.lower().split().count(word) for word in unique_words] for msg in X_train])

In [7]:
prior_spam = np.mean(y_train == "Spam")
prior_not_spam = np.mean(y_train == "Not Spam")

spam_word_count = np.sum(word_matrix[y_train == "Spam"], axis=0)
not_spam_word_count = np.sum(word_matrix[y_train == "Not Spam"], axis=0)

In [8]:
def likelihood(word_count, word):
    if word not in unique_words:
        return 1 / (np.sum(word_count) + len(unique_words))
    return (word_count[unique_words == word][0] + 1) / (np.sum(word_count) + len(unique_words))

likelihood(unseen words)=1/total word count+volcabulary size
P(word∣class)= (word count in class)+1/(total word count in class+vocabulary size)

In [9]:
def predict(sample):
    likelihood_spam = np.prod([likelihood(spam_word_count, word) for word in sample.lower().split() if word not in stopwords and not word.isnumeric()])
    likelihood_not_spam = np.prod([likelihood(not_spam_word_count, word) for word in sample.lower().split() if word not in stopwords and not word.isnumeric()])
    posterior_spam = prior_spam * likelihood_spam
    posterior_not_spam = prior_not_spam * likelihood_not_spam
    evidence = posterior_spam + posterior_not_spam
    posterior_spam /= evidence
    posterior_not_spam /= evidence
    return (posterior_spam, posterior_not_spam, "Spam" if posterior_spam > posterior_not_spam else "Not Spam")

posterior prob=P(class∣sample)∝P(class)⋅P(sample∣class)

In [10]:
test_sample = X_test[2]
test_sample

'Exclusive limited time offer'

In [11]:
predict(test_sample)

(np.float64(0.9638045734148131), np.float64(0.036195426585186846), 'Spam')

In [12]:
for sample in X_test:
    print(f"'{sample}' → {predict(sample)[2]}")

'Win a prize today' → Spam
'Join us for lunch' → Not Spam
'Exclusive limited time offer' → Spam
'Meeting at my place tomorrow' → Not Spam


In [13]:
model = MultinomialNB()
model.fit(X_train_counts, y_train)

predictions = model.predict(X_test_counts)

for doc, label in zip(X_test, predictions):
    print(f"'{doc}' → {label}")

'Win a prize today' → Spam
'Join us for lunch' → Not Spam
'Exclusive limited time offer' → Spam
'Meeting at my place tomorrow' → Not Spam
